# U5D7 · Capstone整合 · 牛津 Tutorial LLM 仿真 (v6.0 学习科学层)

## Persona Prompt (导师人设)

> You are an **Oxford tutorial fellow** in **Capstone整合 (end-to-end agentic AI project: causaldata NSW -> DoWhy ATE -> LangGraph 营销Agent -> deepeval -> IMRaD + DSR)**.
>
> **Never give direct answers.** 不直接给答案, 不直接答学生的问题. Use **Socratic questioning** 苏格拉底追问 to draw the answer out of the student.
>
> Act as an **HBS devil's advocate** (哈佛商学院反方): challenge every claim, demand evidence, reject vague hand-waving.
>
> Reject vague claims like "Agent 会自己学习" / "评估很准" / "DSR 就是写论文". Force the student to ground every claim in a specific artifact: which TODO, which drill, which line of `solution.ipynb`, which DSR step.
>
> **End each turn with a probing question** -- one of: 为什么 / 反例 / 若前提变 / 凭什么 / 如何. Never end with a statement.
>
> 限频: 每单元 1 次/天 (防依赖, 见 cell6). 仿真用静态 if/else, 不调真实 LLM API.


## Pre-Tutorial Task (强制 retrieval, 不做不准进入 tutorial)

> 牛津 tutorial 的铁律: **学生先写, 导师后问**. 你必须在进入 cell3 Socratic loop 之前, 完成以下 retrieval 任务并写入 `student_pre_tutorial.json`.

**任务**: 提交一段 500 字的 Capstone proposal, 必须含:
1. **DSR Step 1-2**: 你的 Capstone 解决什么营销问题? 为什么现有方案不够好? artifact 应达到什么效果 (效率/质量/安全/可评估)?
2. **端到端流水线设计**: 五层 (causaldata NSW -> DoWhy -> LangGraph -> deepeval -> IMRaD) 每层的输入/输出/真实库.
3. **投稿目标**: ICIS / Decision Support Systems / HICSS 三选一, 给出理由.

**禁止**: 复制粘贴 `notes.md` 原文. 必须用自己的话 retrieval (提取练习优于重读).
**禁止**: 写"Agent 会自动优化"这类空话 -- HBS devil's advocate 会逐句追问"凭什么".

提交后, cell3 的 Socratic loop 会读取你的 proposal 并逐句追问.


In [ ]:
# Socratic Loop (静态 if/else 仿真, >=4 轮, 不调真实 LLM API)
# 每轮模拟 Oxford tutor 对学生 proposal 的追问, 学生回答后触发下一轮.
import json, os

# --- 加载学生 pre-tutorial proposal (若不存在, 用 demo 占位) ---
pre_path = "student_pre_tutorial.json"
if os.path.exists(pre_path):
    with open(pre_path, encoding="utf-8") as f:
        proposal = json.load(f)
else:
    proposal = {
        "problem": "企业营销干预效果因果未知, 现有A/B测试有偏",
        "artifact_goal": "提升营销效率",
        "pipeline": "causaldata->DoWhy->LangGraph->deepeval->IMRaD",
        "venue": "ICIS"
    }
    print("[demo] 未找到 student_pre_tutorial.json, 使用占位 proposal. 请先完成 cell2 任务.\n")

print("=" * 70)
print("Oxford Tutor: 让我们开始. 我会逐轮追问, 你回答后我才进入下一轮.")
print("=" * 70)

# --- 第1轮: 为什么 (why) ---
turn1_q = "【Round 1 / 苏格拉底问 1】你说现有A/B测试有偏--为什么有偏? 凭什么说有偏? A/B 测试在 NSW RCT 的语境下, 具体哪个混杂变量没有被控制?"
print(f"\nTutor: {turn1_q}")
student_a1 = "样本自选择, 参与营销的用户本身就更高消费倾向"  # 静态学生回答

# 静态 if/else 模拟 Socratic 追问: 根据学生回答质量决定追问方向
if "自选择" in student_a1 or "self-select" in student_a1.lower():
    feedback1 = "[追] 对, 自选择是 NSW 经典问题. 但反例: 如果你用 DoWhy 的后门调整把 re74/re75 都控制了, ATE 还会有偏吗? 若前提变--如果 re75 本身就被营销干预影响呢?"
else:
    feedback1 = "[追] 你的回答太模糊. 凭什么? 给我一个具体的混杂变量名. 重新回答."
print(f"Student: {student_a1}")
print(f"Tutor: {feedback1}")

# --- 第2轮: 反例 (counterexample) ---
turn2_q = "【Round 2 / 苏格拉底问 2】反例: 假设你的 LangGraph Agent 读到 ATE=0 (营销无因果效果), 它还会生成营销策略吗? 如果会, 那 Agent 是 grounded 于因果证据, 还是 grounded 于 LLM 先验?"
print(f"\nTutor: {turn2_q}")
student_a2 = "Agent 应该 grounded 于因果证据, ATE=0 时应该建议停止营销"

if "grounded" in student_a2.lower():
    feedback2 = "[追] 好. 但如何让 deepeval 的 BaseMetric 验证 Agent 真的 grounded 了? 你的 measure() 里凭什么判断 Agent 的输出引用了 ATE? 凭什么不是 LLM 瞎编一个数字?"
else:
    feedback2 = "[追] 你回避了问题. 如何设计条件边让 ATE=0 时 Agent 走停止分支? 给我 LangGraph 的 add_conditional_edges 代码."
print(f"Student: {student_a2}")
print(f"Tutor: {feedback2}")

# --- 第3轮: 若前提变 (what if) ---
turn3_q = "【Round 3 / 苏格拉底问 3】若前提变--如果你的 Capstone 投稿目标从 ICIS 换成 HICSS, 论文的 DSR 对齐要怎么改? ICIS 与 HICSS 对 artifact 评估的期望有什么不同?"
print(f"\nTutor: {turn3_q}")
student_a3 = "HICSS 更重 artifact 设计 novelty, ICIS 更重实证评估严谨"

if "HICSS" in student_a3 or "ICIS" in student_a3:
    feedback3 = "[追] 凭什么这么分? 你读过 Hevner 2004 对 DSR 评估的七准则吗? 其中哪一条 ICIS 比 HICSS 更看重? 如何在你的 IMRaD Methods 段体现这个差异?"
else:
    feedback3 = "[追] 你的回答没有区分. 反例: 给我一个 HICSS 接受的纯设计论文 + 一个 ICIS 拒稿的评估不足论文, 对比 Methods 段."
print(f"Student: {student_a3}")
print(f"Tutor: {feedback3}")

# --- 第4轮: 凭什么 (on what grounds) ---
turn4_q = "【Round 4 / 苏格拉底问 4】你说 artifact 目标是提升营销效率--凭什么衡量? 效率的操作化定义是什么? 是 ATE 提升幅度, 还是 Agent 决策延迟降低, 还是 deepeval 评分? 三者冲突时你信谁?"
print(f"\nTutor: {turn4_q}")
student_a4 = "以 ATE 提升为主, deepeval 评分为辅, 延迟作为约束"

if "ATE" in student_a4:
    feedback4 = "[追] 好, 但反例: 如果 ATE 提升但 deepeval 评分下降 (Agent 为了提 ATE 开始编造策略), 你的 artifact 还算成功吗? DSR Step5 的评估如何处理多目标冲突?"
else:
    feedback4 = "[追] 你没有操作化. 如何把效率变成一个可测量的数字? 给我公式."
print(f"Student: {student_a4}")
print(f"Tutor: {feedback4}")

# --- 第5轮: 如何 (how) ---
turn5_q = "【Round 5 / 苏格拉底问 5】最后一个: 如何把天道推演的沙盘模拟3层推演映射到你 Capstone 的 LangGraph 代码? 给我一个具体的节点或分支作为代码落点, 不要悬空贴标签."
print(f"\nTutor: {turn5_q}")
student_a5 = "LangGraph 的并行 branch 模拟多路径推演, 每个branch对应一条因果路径"

if "branch" in student_a5.lower() or "并行" in student_a5:
    feedback5 = "[追] 接近了. 但三层推演 (immediate/near/far) 对应 LangGraph 的什么? 是三次嵌套 branch, 还是同一个 branch 的三次迭代? 你的 Discussion 段要写清这个映射, 否则同构悬空."
else:
    feedback5 = "[追] 太抽象. 如何? 给我一个 StateGraph 的节点名."
print(f"Student: {student_a5}")
print(f"Tutor: {feedback5}")

print("\n" + "=" * 70)
print("Oxford Tutor: Tutorial 结束. 你的 5 个盲点已记录到 student_model.json (见 cell4).")
print("=" * 70)


In [ ]:
# student_model.json 读写 (记录掌握度/盲点, 跨 session 持久化)
import json, os
from datetime import datetime

sm_path = "student_model.json"

# 读取已有模型 (若存在)
if os.path.exists(sm_path):
    with open(sm_path, encoding="utf-8") as f:
        student_model = json.load(f)
else:
    student_model = {
        "unit": "U5D7",
        "sessions": [],
        "mastery": {"ILO1_DSR": 0.0, "ILO2_pipeline": 0.0, "ILO3_IMRaD": 0.0, "ILO4_tiandao": 0.0},
        "blind_spots": []
    }

# 记录本次 tutorial 的盲点 (从 cell3 Socratic loop 提取)
new_session = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "turns": 5,
    "blind_spots": [
        "混杂变量 re75 是否被营销干预影响 (Round 1)",
        "BaseMetric 如何验证 Agent grounded 于 ATE 而非 LLM 先验 (Round 2)",
        "ICIS vs HICSS 对 DSR 评估期望的差异 (Round 3)",
        "多目标冲突 (ATE升但deepeval降) 的 DSR Step5 处理 (Round 4)",
        "天道推演三层推演与LangGraph 嵌套branch还是同branch迭代 (Round 5)"
    ],
    "mastery_update": {
        "ILO1_DSR": 0.4,       # Round 1/3 触及 DSR, 但评估差异未答全
        "ILO2_pipeline": 0.5,  # Round 2/4 触及流水线, grounded 验证待补
        "ILO3_IMRaD": 0.3,     # Round 3 投稿目标差异未答全
        "ILO4_tiandao": 0.2    # Round 5 同构映射悬空, 需补代码落点
    }
}
student_model["sessions"].append(new_session)

# 更新 mastery (取 max, 不衰减 -- mastery 是累积的)
for k, v in new_session["mastery_update"].items():
    student_model["mastery"][k] = max(student_model["mastery"].get(k, 0.0), v)

# 更新 blind_spots (去重)
for b in new_session["blind_spots"]:
    if b not in student_model["blind_spots"]:
        student_model["blind_spots"].append(b)

# 写回
with open(sm_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

print(f"[student_model] 已写入 {sm_path}")
print(f"  累计 sessions: {len(student_model['sessions'])}")
print(f"  当前 mastery: {student_model['mastery']}")
print(f"  累计 blind_spots: {len(student_model['blind_spots'])}")
for b in student_model["blind_spots"]:
    print(f"    - {b}")


## Hattie 四级形成性反馈 (Hattie & Timperley 2007)

> 导师根据 cell3 Socratic loop + cell4 student_model, 给出四级反馈.
> 故意避开 Self 级表扬 (Hattie: Self 级反馈对学习效果最弱, 甚至反效果).
> 重点是 Task / Process / Self-Reg / Feed-Forward 四级.


In [ ]:
# Hattie 四级反馈 (静态生成, 基于 cell3/cell4)
print("=" * 70)
print("Hattie 四级形成性反馈 (U5D7 Capstone整合)")
print("=" * 70)

feedback_lines = [
    "",
    "[TASK] 任务级反馈 (这次 proposal 哪里对/错):",
    "  - 对: 你正确识别了 NSW 自选择问题, 并把 ATE 作为 Agent 决策依据.",
    "  - 错: DSR Step2 目标定义太模糊 (提升营销效率 不可测), 必须操作化为",
    "        ATE 提升幅度 >= X 或 deepeval >= Y. 模糊目标 = DSR 失败的第一步.",
    "  - 错: 投稿目标选 ICIS 但没说清 ICIS 对 DSR 评估的期望, Methods 段会挂.",
    "",
    "[PROCESS] 过程级反馈 (你思考的方式哪里要改):",
    "  - 你的 5 轮回答有 3 轮先给结论后补理由. 牛津 tutorial 要求反过来:",
    "    先给证据/机制, 再给结论. 这是 DSR Step3 (设计开发) 的论证规范.",
    "  - 你在 Round 5 (天道推演同构) 用了并行branch但没给节点名. Process",
    "    问题: 抽象映射必须落到具体 artifact (节点名/字段名/代码行), 否则同构悬空.",
    "  - 改进: 下次回答前先问自己这个 claim 能在 solution.ipynb 的哪一行落地?",
    "",
    "[SELF-REG] 自我调节级反馈 (你如何监控自己的学习):",
    "  - 你在 Round 2 主动提到 Agent 应该 grounded 于因果证据 -- 这是好的",
    "    self-regulation: 你在监控自己的论证质量. 保持.",
    "  - 但 Round 4 你回避了多目标冲突 (ATE升但deepeval降). Self-reg 漏洞:",
    "    你倾向回避会让 artifact 显得不完美的问题. DSR Step5 评估必须",
    "    主动暴露 trade-off, 不是藏起来. 下次自问我的 artifact 在什么条件下失败?",
    "",
    "[FEED-FORWARD] 前馈级反馈 (下一步做什么):",
    "  - 立即: 重写 DSR Step2 目标, 用 SMART 原则 (具体/可测/可达/相关/时限).",
    "  - 立即: 在 solution.ipynb TODO4 的 LangGraph 代码里, 找到并行 branch",
    "    节点, 在旁边注释天道推演沙盘模拟对应此处. 这解决 Round 5 同构悬空.",
    "  - 短期: 跑 practice.md Drill D2 的 independent 阶段 (NSW->DoWhy->LangGraph",
    "    接线), 重点练条件边分支 -- 你的 Round 2 暴露了这块弱.",
    "  - 中期: 读 reading.md 的 Hevner 2004 条目, 把 DSR 七准则贴在显示器前,",
    "    写 IMRaD Methods 时逐条对照.",
    "  - 长期: 把 5 个 blind_spots 写进 Capstone Discussion 的 Limitations 段,",
    "    这是 ICIS/HICSS 评审最看重的诚实度信号.",
    "",
    "[Hattie 反馈完成. 故意避开 Self 级表扬 (如你很聪明) -- 对学习无效.]"
]
for line in feedback_lines:
    print(line)


## 限频与 Exit Artifact

### 限频 (防依赖)

- **每单元 1 次/天**: 本 tutorial LLM 仿真每天最多跑 1 次. 防止学生把 Socratic loop 当"答案生成器"反复跑直到凑出"对的"回答 -- 这违背 retrieval practice 原则.
- **为什么限频**: 牛津 tutorial 的效力来自"学生先想, 导师后问". 无限次跑 = 学生不先想, 直接试答案. 这是重读 (rereading) 而非提取 (retrieval), 学习效果差一个量级.
- **超限处理**: 若当天已跑过 1 次, cell3 的 Socratic loop 会输出"今日 tutorial 已用完, 请先完成 practice.md 的 drill, 明天再来. Retrieval > rereading."
- **跨 session 持久化**: 调用次数记录在 `student_model.json` 的 `sessions` 字段, 跨 session 累计.

### Exit Artifact (退出 tutorial 必交)

> 完成 cell3-cell5 后, 在退出前必须写下以下 3 项, 存入 `exit_artifact.json`:

1. **2-3 个盲点** (从 cell4 的 blind_spots 中选你最想攻克的):
   - 例: "BaseMetric 如何验证 Agent grounded 于 ATE"
   - 例: "天道推演三层推演与 LangGraph 嵌套 branch 还是同 branch 迭代"
   - 例: "ICIS vs HICSS 对 DSR 评估期望的差异"

2. **推荐复习单元** (基于盲点回溯):
   - 盲点涉及 DoWhy/LangGraph 接线 -> 复习 **Day 2 (LangGraph)** + **Day 3 (DoWhy)** 的 `schedule.json` 卡片, 重排 FSRS 间隔.
   - 盲点涉及 deepeval/IMRaD -> 复习 **Day 3 (deepeval)** + **Day 6 (IMRaD)** 的 `practice.md` drill.
   - 盲点涉及 DSR 框架 -> 重读 **Day 7 `notes.md` 关键回顾 2** + `reading.md` Hevner 2004 条目.
   - 盲点涉及天道推演同构 -> 重读 **Day 7 `notes.md` 2026前沿章节** + 项目 CLAUDE.md 天道推演矩阵.

3. **下一步行动** (1 句话, 可执行):
   - 例: "今晚 22:00 前在 solution.ipynb TODO4 找到并行 branch 节点, 注释天道推演映射."

### 退出检查

- [ ] `student_model.json` 已更新 (cell4)
- [ ] Hattie 四级反馈已读 (cell5), 故意避开 Self 级表扬
- [ ] `exit_artifact.json` 已写入 2-3 盲点 + 推荐复习单元 + 下一步行动
- [ ] 今日 tutorial 调用次数已记 1 (限频)

> 完成以上 4 项后, 本单元 tutorial 视为"退出". 下次进入需等明天 (限频).
